# ALGORITHMIC TRADING FINAL PROJECT

- project abstract

## 1. Project Setup

### 1.1 Trading Platform

#### Broker Selection Decision

After completing the initial exploration of forex trading platforms, several brokers were evaluated for automated trading in Germany. The criteria for selection were full regulation according to EU/BaFin standards, comprehensive Python API access, and a wide range of trading instruments. The choice fell on Interactive Brokers, which provides a tested platform for real futures contracts, has excellent Python API support (via the ib_async library), offers competitive fees and a wide instrument range, and is fully regulated in Germany.

Interactive Brokers (IBKR) is a global online brokerage firm headquartered in the United States. It provides direct access to a wide range of financial markets and instruments, including stocks, options, futures, bonds, and foreign exchange. It is known for its strong technological focus, low trading costs, and infrastructure that not only fulfills the needs of professional and institutional participants but is also suitable for retail traders.

The company was founded by Thomas Peterffy, one of the early pioneers of algorithmic trading. Starting as an architectural draftsman working on highway projects, he left this career path to trade equity options and apply his strategies to computerized trading models. He developed some of the first automated execution systems, thereby laying the groundwork for modern electronic markets. This technology-driven background is still visible today in Interactive Brokers' powerful trading platform and extensive API support.

For traders based in Europe—and especially in Germany—Interactive Brokers is a practical and reliable choice. The broker operates through regulated European entities and offers broad access to international markets. Compared to many alternatives, it complies with local financial regulations and provides competitive fees and transparent pricing. The main drawback is that it requires expensive data subscriptions for live streaming data of certain asset classes like stocks.

Features such as fully functional paper trading accounts and robust API connectivity make IBKR particularly well suited for algorithmic and systematic trading projects. Interactive Brokers combines regulatory reliability, global market access, and strong technical capabilities, making it a natural choice for quantitative and algorithmic trading workflows.

#### The GUI Requirement Challenge

The key technical challenge with Interactive Brokers is that API access requires a GUI environment, either Trader Workstation (TWS) or IB Gateway. This approach presents several problems:

- Headless servers: Cloud instances don't have display environments
- 24/7 operations: Running locally requires keeping a computer on continuously
- Remote access: Difficult to access from different locations
- Stability: Interruptions of local networks affect trading operations

The solution that aligns with this project's requirements for a fully automated trading system was to deploy IB Gateway in a Docker container.

#### Cloud Instance Requirements Analysis

For IB Gateway deployment in a Docker container, the following requirements were identified:

__Technical Requirements:__
- Linux OS (Ubuntu preferred for compatibility)
- Sufficient RAM for Java process (IB Gateway is Java-based)
- Stable network connection
- Docker support

__Cost Considerations:__
- Must be more economical than running a local computer 24/7
- Predictable monthly costs
- Included bandwidth for trading data

__Geographic Considerations:__
- Low latency to Interactive Brokers servers
- European datacenter preferred (trading from Germany)

__Specifications of Selected Configuration (DigitalOcean Droplet):__
```
OS:          Ubuntu 22.04 LTS
Plan:        Basic $12/month
Resources:   2 GB RAM / 1 CPU / 50 GB SSD
Datacenter:  Frankfurt, Germany
Transfer:    2 TB included
````
Ubuntu 22.04 LTS was chosen for its long-term support, excellent Docker compatibility, and extensive documentation. Since Java processes can be memory-intensive, the 1 GB RAM option was deemed insufficient, leading to the selection of 2 GB RAM.

Frankfurt was selected as the datacenter location because, being located in Germany, a datacenter with the lowest latency from Germany and proximity to IB's European servers was needed. The $12/month tier provides reasonable costs while guaranteeing better performance and stability — much cheaper than running a computer or laptop at home 24/7.

### 1.2 Initial Server Setup

#### Phase 1: Droplet Creation

After setting up a DigitalOcean account, a droplet needs to be created with the previously described configuration. This involves:

- Selecting Ubuntu 22.04 LTS image
- Choosing Basic plan with 2 GB RAM
- Selecting Frankfurt datacenter
- Adding SSH public key for authentication
- Naming droplet: ib-gateway-server

To connect to the droplet from a local machine, an SSH key needs to be generated first:
```
# Generate SSH key on Mac terminal
ssh-keygen -t ed25519 -C "your_email@example.com"
   
# Display public key
cat ~/.ssh/id_ed25519.pub
```

Ed25519 is a modern, high-security digital signature algorithm based on elliptic curve cryptography, specifically designed for speed, safety, and resistance to common implementation flaws. It's faster and more secure than older RSA keys, making it the recommended choice for SSH authentication.

__Note:__ In the following descriptions, the actual IP address of the cloud instance (provided by DigitalOcean) will be replaced with XXX.XXX.XXX.XX for security reasons.

#### Phase 2: Initial Server Configuration

Once the droplet is created and accessible, the server needs to be secured and prepared for Docker deployment.

__1. Connecting to droplet via SSH:__
```
# Connect to new droplet
ssh root@XXX.XXX.XXX.XX
```

__2. Updating the operating system:__
```
# Update package lists
apt update
   
# Upgrade installed packages
apt upgrade -y
   
# Install useful tools
apt install -y curl wget vim net-tools
```

__3. Setting up a firewall:__
```
# Install firewall
apt install -y ufw
   
# Allow SSH (critical - don't lock yourself out!)
ufw allow 22/tcp
   
# Enable firewall
ufw enable
   
# Check status
ufw status
```
Expected output:
```
Status: active

To                         Action      From
--                         ------      ----
22/tcp                     ALLOW       Anywhere
````
Initially, restricting SSH access to a specific IP address was considered. However, with a dynamic home IP, keeping SSH open with key authentication proved to be a more practical solution while still maintaining strong security.

#### Phase 3: Docker Installation

Docker is required to run IB Gateway in a containerized environment, providing isolation and easy management of the application.

__1. Installing Docker:__
```
# Download Docker installation script
curl -fsSL https://get.docker.com -o get-docker.sh
   
# Run installation script
sh get-docker.sh
   
# Verify installation
docker --version
```
Expected output: `Docker version 24.0.x, build xxxxxxx`

__2. Testing Docker__
```
# Run test container
docker run hello-world
```
If Docker is properly installed, this command will download a test image and display a "Hello from Docker!" message confirming that the installation is working correctly.

### 1.3 IB Gateway Container Deployment

#### Phase 1: Docker Image Selection

After researching available IB Gateway Docker images, `ghcr.io/unusualalpha/ib-gateway:latest` was found to be the most suitable and robust option. The selected image:

- Is the most popular and actively maintained
- Handles xvfb (virtual display) automatically
- Includes IBC (Interactive Brokers Controller) for automation
- Is well-documented and widely used in the community
- Has regular updates to match IB Gateway releases

#### Phase 2: Credentials Configuration

__1. Creating configuration directory:__
```
mkdir -p ~/ib-config
cd ~/ib-config
```

__2. Creating credentials file:__
```
nano ib-credentials.env
```
File contents of `ib-credentials.env`:
```
TWS_USERID=your_ib_username
TWS_PASSWORD=your_ib_password
TRADING_MODE=paper
VNC_SERVER_PASSWORD=your_vnc_password
```
Explanation of parameters:
- TWS_USERID: Interactive Brokers username
- TWS_PASSWORD: Interactive Brokers password
- TRADING_MODE: paper for paper trading, live for real trading
- VNC_SERVER_PASSWORD: Password for VNC remote desktop access

__3. Securing credentials file:__
```
# Set restrictive permissions (owner read/write only)
chmod 600 ib-credentials.env
   
# Verify permissions
ls -la ib-credentials.env
# Should show: -rw------- 1 root root
```

#### Phase 3: Container Deployment

__1. Pulling the Docker image:__
```
docker pull ghcr.io/unusualalpha/ib-gateway:latest
```

__2. Running the IB Gateway container:__
```
docker run -d \
     --name ib-gateway \
     --restart unless-stopped \
     -p 4001:4001 \
     -p 4002:4002 \
     -p 5900:5900 \
     --env-file ~/ib-config/ib-credentials.env \
     ghcr.io/unusualalpha/ib-gateway:latest
```
Breakdown of Docker commands:
- `-d`: Run in detached mode (background)
- `--name ib-gateway`: Name the container for easy reference
- `--restart unless-stopped`: Auto-restart on failures or system reboot
- `-p 4001:4001`: Paper trading API port
- `-p 4002:4002`: Live trading API port
- `-p 5900:5900`: VNC port for GUI access
- `--env-file`: Load credentials from secure file

__3. Verifying container is running:__
```
# Check container status
docker ps
   
# Should show ib-gateway with STATUS "Up"
```

__4. Monitoring startup logs:__
```
# View logs
docker logs ib-gateway
   
# Follow logs in real-time
docker logs -f ib-gateway
```
Expected log messages:

- "IBC: Starting Gateway"
- "IBC: Detected frame entitled: IBKR Gateway"
- "IBC: Login attempt"
- "IBC: Login has completed"

#### Phase 4: Initial Configuration via VNC

At this stage of the configuration process, first-time paper trading accounts require the configuration of certain API settings through the GUI interface. To accomplish this, it is necessary to connect to the IB Gateway instance running on the droplet from a local computer (in this case, a Mac) via VNC.

__1. Connecting to VNC from local Mac:__
- Opening Finder
- Pressing Cmd+K
- Entering server address: vnc://XXX.XXX.XXX.XX:5900
- Entering VNC password (from credentials file)

__2. Verify API Settings:__
Once connected, navigate to: Configure → Settings → API → Settings
- Verify the following settings:
  - Socket port: 4002 (for live) or 4001 (for paper)
  - Trusted IPs includes 127.0.0.1
  - "Read-Only API" is _unchecked_
  - "Allow connections from localhost only" is _unchecked_

#### Phase 5: Connection Architecture and SSH Tunnel

__The IPv4/IPv6 Challenge__  
Initial attempts to connect directly from Mac to droplet failed due to IPv4/IPv6 networking issues between Docker, the host OS, and the remote client.

__Technical Details:__
- IB Gateway Java process was binding to IPv6 (:::4002)
- Mac's ib_async was attempting IPv4 connection
- Docker bridge networking added complexity
- Firewall considerations created additional challenges

__SSH Tunnel Solution__  
After researching solutions, SSH tunneling emerged as the recommended approach for remote IB Gateway access. This method not only bypasses networking issues but also provides enhanced security compared to direct connections.

__How It Works:__
```
Local Mac:4002 → SSH Tunnel → Server:22 → Server:localhost:4002 → IB Gateway
```

__Implementation:__

__1. Creating an SSH tunnel on local computer__
```
# -4 forces IPv4
# -L creates local port forwarding
ssh -4 -L 4002:127.0.0.1:4002 root@XXX.XXX.XXX.XX
```

The terminal window where the tunnel is running needs to be kept open. Closing it terminates the tunnel.

__2. Connecting to the tunnel from a Python script:__
```
# Connect to localhost (tunneled to server)
from ib_async import *

# Create IB connection object
ib = IB()
ib.connect('127.0.0.1', 4002, clientId=1, timeout=20)

# Verify connection
print(f"Connected: {ib.isConnected()}")
```
Expected output:
```
Connected: True
```

__Security Benefits of SSH Tunnel:__
The SSH tunnel approach provides security advantages that make it superior to direct port exposure. First, all communication between the local machine and the server is encrypted through the SSH protocol, protecting sensitive trading data and credentials from interception. This encryption applies to the entire data stream, including API commands and market data.

Second, the tunnel eliminates the need to expose IB Gateway ports (4001, 4002) directly to the internet. Instead, only the SSH port (22) needs to be accessible, significantly reducing the attack surface. Any attempt to access the trading API must first authenticate through SSH using key-based authentication, adding an additional security layer that port-based connections lack.

Third, this architecture provides flexibility regardless of network configuration. Whether dealing with IPv4/IPv6 issues, NAT traversal, or Docker networking complexities, the SSH tunnel creates a reliable connection path. This is why SSH tunneling is considered standard practice among professional trading firms for remote gateway access—it combines robust security with practical reliability.

__3. Final Firewall Configuration:__
```
# Only SSH port exposed
ufw status
```
Expected output:
```
Status: active

To                         Action      From
--                         ------      ----
22/tcp                     ALLOW       Anywhere
22/tcp (v6)                ALLOW       Anywhere (v6)
```

#### Conclusion

This deployment successfully established a production-grade infrastructure for automated trading. The IB Gateway now runs continuously on the DigitalOcean droplet, independent of any local computer, providing true 24/7 operation. The multi-layered security architecture—combining SSH key authentication, firewall restrictions, and encrypted tunneling—ensures that trading operations are protected while remaining accessible from any location with SSH access.

The financial benefits are substantial: at \\$12 per month, the cloud-based solution costs significantly less than running a local computer continuously, which would consume \\$50-100 monthly in electricity and wear. Docker's automatic restart capability ensures the gateway remains operational even after system updates or unexpected failures, eliminating the need for manual intervention.

Perhaps most importantly, this architecture is not merely a proof-of-concept but represents the same approach used by professional trading firms. The combination of containerization, secure remote access, and reliable cloud infrastructure creates a foundation suitable for both development and production trading environments. The system is now ready to support automated trading strategies with the reliability and security that real-world trading demands.

## 2. Strategic Decisions

### 2.1 Choice of Instrument

As the trading instrument, the EUR/USD currency pair is chosen. The focus is on direct foreign exchange trading rather than CFDs, because this allows to put a greater emphasis on macroeconomic factors and geopolitical events that have a direct influence on currency values. Given the current political situation between Europe and the United States, this currency pair is expected to show a high level of unpredictability and volatility, making it particularly interesting to trade in the near future. From a practical perspective, forex trading is available immediately within my Interactive Brokers' account setup and does not require costly subscriptions for live market data.

### 2.2 Trading Data

For the backtesting task, historical data for the EUR/USD currency pair is obtained from Interactive Brokers with the EClient.reqHistoricalData API method.

***[more text needs to be added explaining the choice of different timeframes]***

### 2.3 Trading Strategy

## 3. Data Preparation

### 3.1 Imports

### 3.2 Data Extraction

### 3.3 Data Preparation

### 3.4 Data Quality Assessment

### 3.5 Data Limitations and Considerations

## 4. Technical Indicator Calculation

### 4.1 Indicator Calculation

### 4.2 Indicator Summary

## 5. Signal Generation and Position Management

### 5.1 Understanding Signal Generation

### 5.2 Generating Trading Signals based on Indicator Combination

### 5.3 Converting Signals to Actual Positions

### 5.4 Signal Validation

## 6. Backtest Implementation

### 6.1 What Questions Will Backtesting Answer?

### 6.2 Strategy Readiness Assessment

### 6.3 Backtesting Methodology

### 6.4 Setting Backtest Parameters

### 6.5 Calculating Returns with Transaction Costs

### 6.6 Tracking Portfolio Value Over Time

### 6.7 Analyzing Individual Trades

### 6.8 Risk-Adjusted Performance Metrics

- Sharpe Ratio:
- Sortino Ratio:
- Maximum Drawdown:
- Calmar Ratio:

### 6.9 Comparing to Buy-and-Hold

### 6.10 Visualizing Strategy Performance

### 6.11 Backtest Results Summary

## 7. Implementing Live (Paper) Trading with Interactive Brokers

### 7.1 Overview

To transition from backtesting to real-time execution, live streaming market data from Interactive Brokers is required. In this section, I describe the implementation of a fully automated paper trading system for the EUR/USD currency pair using the Interactive Brokers Gateway. All trades are executed on a paper trading account; therefore, whenever “live trading” is mentioned in the following, it refers to simulated trading under real market conditions but with virtual capital.

The IB Gateway is already running on a cloud instance hosted at DigitalOcean (see 1. Project Setup). To enable live trading, the trading logic developed for the backtesting system must be deployed to the same server environment. This deployment was approached in three stages to ensure system reliability.


#### Stage 1: Local Development and Testing
The trading system got developed and tested on a local machine, connecting to the remote IB Gateway via SSH tunnel. This configuration allowed rapid iteration during development while streaming live market data and executing paper trades.

#### Stage 2: Extended Validation Run
After verifying stable operation over several hours, a 5-hour production run was conducted to validate long-term stability. This test remained local to identify any issues that might only emerge during extended operation before committing to cloud deployment.

#### Stage 3: Cloud Deployment
The final deployment stage consisted of moving the trading bot directly onto the cloud instance alongside the IB Gateway. This configuration eliminates network dependencies and provides maximum reliability for continuous automated trading.

This live trading system transitions the strategy from backtesting with historical data to real-time execution while preserving the same trading logic. At the same time, it introduces additional components required for handling live market data, order execution, and position management.

The implementation follows a modular architecture consisting of ***[six core Python modules, each handling specific aspects of the trading workflow. This separation of concerns enables easier testing and debugging and provides a flexible foundation for future extensions]***.

### 7.2 Preliminary Considerations for the Project

Several practical constraints of the Interactive Brokers account need to be taken into account when operating the live trading system. Only one position per instrument can be traded at any given time, meaning that long and short positions cannot be open simultaneously. In addition, the base currency of the trading account is Euro and is not automatically converted into USD for trading purposes.

Before each execution of the trading program, it is therefore necessary to check if a sufficient amount of USD cash balance is available to meet the minimum contract size for EUR/USD (20,000 units). Furthermore, existing open positions must be checked and, if necessary, closed or reversed before opening a new opposite position.

### 7.3 Requirements for the Live Trading Architecture

The live trading system is designed to satisfy the following functional requirements:

- Real-time data streaming using XXX bars
- Live indicator calculation (moving averages, RSI, momentum) on each new bar
- Signal generation using the same logic as for the backtest
- Order execution via market orders submitted to Interactive Brokers
- Position tracking to maintain an accurate view of open positions
- Trade logging with detailed records stored in CSV format
- Performance monitoring with tracking P&L real-time
- Safe shutdown procedure, including clean termination and graceful handling of market closures
- Automatic reconnection capability to handle network interruptions during long runs

### 7.4 Revisiting the Trading Strategy

### 7.5 System Architecture

### 7.6 Implementation Challenges and Solutions

***[maybe not necessary]***

### 7.7 Testing and Validation

## 8. Cloud Deployment

After successfully developing and testing the trading system locally, the next step was to deploy it to a cloud environment for continuous autonomous operation. The DigitalOcean droplet already had the IB Gateway running in a Docker container, providing the connection to Interactive Brokers' paper trading environment. The goal was to deploy the trading bot in a separate Docker container on the same droplet, ensuring it could reliably connect to the IB Gateway and execute trades independently.

### 8.1 Why Cloud Deployment Matters

Running the trading bot locally presented several practical limitations that needed to be overcome. The most obvious constraint was that I couldn't close my laptop or lose internet connectivity without terminating the trading session. For a system designed to operate continuously during market hours, this dependency on my personal computer was unacceptable. Additionally, executing trades from residential internet connections introduces latency and reliability concerns compared to cloud infrastructure with direct network paths to broker servers.

### 8.2 Infrastructure Architecture

The deployment architecture consists of two Docker containers running on the same DigitalOcean droplet:

IB Gateway Container: Already deployed and running, providing API access to Interactive Brokers on port 4002
Trading Bot Container: The newly deployed trading system that connects to IB Gateway

Both containers use Docker's `--network host` mode, which allows them to share the host's network stack. This configuration simplifies connectivity as the trading bot can connect to IB Gateway using `localhost:4002`, just as it does in the local development environment via the SSH tunnel.

### 8.3 Containerization with Docker

#### Dockerfile 
The Dockerfile defines the container environment for the trading bot:

```
FROM python:3.11-slim

WORKDIR /app

# Install system dependencies for ib_insync
RUN apt-get update && apt-get install -y \
    gcc \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy trading bot code
COPY trading_bot.py .
COPY config_live.py .

# Create directory for logs (will be mounted as volume)
RUN mkdir -p /app/logs

# Run the trading bot
CMD ["python", "-u", "trading_bot.py"]
```

Key design decisions:

- __Python 3.11-slim:__ Matches local development environment while minimizing image size
- __gcc installation:__ Required for compiling some Python package dependencies
- __Unbuffered Python (`-u` flag):__ Ensures logs appear in real-time without buffering
- __Logs directory:__ Created for mounting as a volume to persist data outside the container

#### requirements.txt
The dependency file specifies all Python packages needed:
```
ib_async==2.1.0
pandas>=2.0.0
numpy>=1.24.0
```
Note: The package is ib_async version 2.1.0, which is the correct package name for the ib-insync library.

#### .dockerignore:
To keep the build context clean and efficient:
```
__pycache__
*.pyc
*.pyo
*.log
logs/
.git
.gitignore
*.md
```

### 8.4 Deployment Process

The deployment process consists of several sequential steps, executed either on the local machine or on the cloud instance.

#### Step 1: Prepare Local Files [LOCAL]
Before deployment, ensure all necessary files are present in the local project directory:

- `trading_bot.py` - Main trading logic
- `config_live.py` - Configuration file with IB_CLIENT_ID and other parameters
- `requirements.txt` - Python dependencies
- `Dockerfile` - Container definition
- `.dockerignore` - Build exclusions

#### Step 2: Transfer Files to Droplet [LOCAL]
Transfer all necessary files from the local machine to the DigitalOcean droplet:
```
# Ensure the target directory exists
ssh root@XXX.XXX.XXX.XX "mkdir -p /root/trading_bot"

# Transfer Python files and dependencies
scp trading_bot.py config_live.py requirements.txt root@XXX.XXX.XXX.XX:/root/trading_bot/

# Transfer Docker configuration files
scp Dockerfile .dockerignore root@XXX.XXX.XXX.XX:/root/trading_bot/
```

#### Step 3: Build Docker Image [CLOUD]
SSH into the droplet and build the Docker image:
```
# Connect to droplet
ssh root@157.230.113.17

# Navigate to project directory
cd /root/trading_bot

# Build the Docker image
docker build -t trading-bot:latest .

# Verify the image was created
docker images | grep trading-bot
```
The build process downloads the base Python image, installs dependencies, and packages the trading bot code into a Docker image. This typically takes 1-2 minutes on the first build.

#### Step 4: Run the Trading Bot Container [CLOUD]
Launch the trading bot in a Docker container with appropriate configuration:
```
bashdocker run -d \
  --name trading-bot-2h \
  --network host \
  --restart unless-stopped \
  -v /root/trading_bot/logs:/app/logs \
  -v /root/trading_bot/config_live.py:/app/config_live.py \
  trading-bot:latest
```
Docker run flags explained:

- `-d`: Run in detached mode (background process)
- `--name trading-bot-2h`: Assign a meaningful name for easy reference
- `--network host`: Share the host's network stack (enables localhost:4002 connection to IB Gateway)
- `--restart unless-stopped`: Automatically restart the container if it crashes or after system reboot
- `-v /root/trading_bot/logs:/app/logs`: Mount logs directory to persist trading records outside the container
- `-v /root/trading_bot/config_live.py:/app/config_live.py`: Mount configuration file for easy updates without rebuilding

#### Step 5: Verify Successful Deployment [CLOUD]
Check that the container is running and connecting to IB Gateway:
```
# Check container status
docker ps | grep trading-bot

# View real-time logs to confirm connection
docker logs -f trading-bot-2h
```

Expected log output confirming successful connection:
```
Connected to IB Gateway at localhost:4002
Client ID: 753
Account: DU123456
Starting trading session...
Fetching initial market data...
```

Exit the log view with `Ctrl+C` and disconnect from the droplet:
```
exit
```

### 8.5 Confirming Connection to IB Gateway

The successful connection to the IB Gateway is critical for the trading bot's operation. Here's how the connection is established and verified:

__Network Configuration__
Both containers (IB Gateway and trading bot) use Docker's `--network host` mode. This configuration means:

- The containers share the host's network namespace
- No port mapping or Docker bridge networking is involved
- The trading bot connects to IB Gateway using localhost:4002
- This is identical to how the local development environment connects

__Connection Parameters__
In `config_live.py`, the connection parameters are configured:
```
IB_HOST = "localhost"
IB_PORT = 4002  # Paper trading port
IB_CLIENT_ID = 753  # Unique ID for cloud bot
```

The client ID (753) is different from local development IDs to allow simultaneous connections from both local and cloud instances if needed.

#### Verification Steps
To confirm the trading bot successfully connected to IB Gateway:

__1. Check IB Gateway Status [CLOUD]__  
```
ssh root@157.230.113.17 "docker ps | grep ib-gateway"
```
This should show the IB Gateway container is running:
```
0d5f489eafaa   ghcr.io/gnzsnz/ib-gateway:latest   ...   Up 13 hours   ib-gateway
```

__2. View IB Gateway Logs [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX "docker logs ib-gateway | tail -20"
```
Look for connection acceptance messages when the trading bot starts.

__3. Check Trading Bot Connection [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h | head -20"
```
Should show:
```
Connected to IB Gateway at localhost:4002
Client ID: 753
Starting market data stream...
```

### 8.6 Remote Monitoring

One of the primary benefits of cloud deployment is the ability to monitor the trading bot remotely without maintaining an active connection. The following commands enable comprehensive monitoring from any location.

__Basic Status Checks [LOCAL]__  
```
# Check if the bot container is running
ssh root@XXX.XXX.XXX.XX "docker ps | grep trading-bot"

# Check container resource usage
ssh root@XXX.XXX.XXX.XX "docker stats --no-stream trading-bot-2h"
```

__Real-Time Log Monitoring [LOCAL]__  
```
# View live log output (follows new logs as they appear)
ssh root@157.230.113.17 "docker logs -f trading-bot-2h"
```
Use `Ctrl+C` to exit the live log view.

__Specific Information Queries [LOCAL]__  
```
# Check most recent P&L
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'Cumulative P&L' | tail -1"

# Count total trades executed
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'TRADE EXECUTED' | wc -l"

# View recent trading signals
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'Signal:' | tail -10"

# Check for any errors
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep -i error"

# Check time remaining (if bot has fixed duration)
ssh root@XXX.XXX.XXX.XX "docker logs trading-bot-2h 2>&1 | grep 'minutes remaining' | tail -1"
````

__Monitoring Session Example__  
A typical monitoring session from the local machine:
```
# Quick status check
$ ssh root@157.230.113.17 "docker ps | grep trading-bot"
abc123def456   trading-bot:latest   ...   Up 45 minutes   trading-bot-2h

# Check latest P&L
$ ssh root@157.230.113.17 "docker logs trading-bot-2h 2>&1 | grep 'Cumulative P&L' | tail -1"
2026-01-23 10:34:12 - Cumulative P&L: $-2.45 | Open P&L: $1.20

# Count trades so far
$ ssh root@157.230.113.17 "docker logs trading-bot-2h 2>&1 | grep 'TRADE EXECUTED' | wc -l"
18
```

### 8.7 Downloading Results

After the trading session completes, results can be downloaded from the droplet to the local machine for analysis.

__Create Local Results Directory [LOCAL]__  
```
# Create a dated results folder
mkdir -p ~/trading_results_$(date +%Y%m%d)
```

__Download All Result Files [LOCAL]__  
```
# Download CSV trade log
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_*.csv ~/trading_results_$(date +%Y%m%d)/

# Download full text log
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trading_bot_*.log ~/trading_results_$(date +%Y%m%d)/

# Download summary file
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_*_summary.txt ~/trading_results_$(date +%Y%m%d)/
```

__Alternative: Download Specific Files [LOCAL]__  
If you know the exact timestamp of your trading session:
```
# Example with specific timestamp (20260123_094904)
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_20260123_094904.csv ~/trading_results_20260123/
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trading_bot_20260123_094904.log ~/trading_results_20260123/
scp root@XXX.XXX.XXX.XX:/root/trading_bot/logs/trades_20260123_094904_summary.txt ~/trading_results_20260123/
```

### 8.8 Container Lifecycle Management

__Stopping the Bot [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX
docker stop trading-bot-2h
exit
```

__Starting a Stopped Container [CLOUD]__  
```
ssh root@XXX.XXX.XXX.XX
docker start trading-bot-2h
exit
```

__Removing the Container [CLOUD]__  
After downloading results and verifying the data, the container can be removed:
```
ssh root@XXX.XXX.XXX.XX
docker stop trading-bot-2h
docker rm trading-bot-2h
exit
```

__Deploying a New Instance [CLOUD]__  
```
# 1. Update configuration file on droplet
ssh root@XXX.XXX.XXX.XX
nano /root/trading_bot/config_live.py
# (Make changes, e.g., RUN_DURATION = "8 h")
# Save and exit (Ctrl+X, Y, Enter)

# 2. Run new container with updated config
docker run -d \
  --name trading-bot-8h \
  --network host \
  --restart unless-stopped \
  -v /root/trading_bot/logs:/app/logs \
  -v /root/trading_bot/config_live.py:/app/config_live.py \
  trading-bot:latest

exit
```

### 8.9 Deployment Best Practices

Based on the deployment experience, several best practices emerged:

#### 1. Version Control
Keep local copies of all deployment files and maintain version control:
```
git add Dockerfile requirements.txt trading_bot.py config_live.py
git commit -m "Deployment configuration for 2-hour test run"
```

#### 2. Configuration Management
Use volume mounts for configuration files rather than building them into the image. This allows quick parameter changes without rebuilding:
```
-v /root/trading_bot/config_live.py:/app/config_live.py
```

#### 3. Log Persistence
Always mount the logs directory to ensure trading data persists outside the container:
```
-v /root/trading_bot/logs:/app/logs
```

#### 4. Container Naming
Use descriptive container names that indicate the run parameters:
```
--name trading-bot-2h   # For 2-hour run
--name trading-bot-8h   # For 8-hour run
```

#### 5. Monitoring Scripts
Create local shell scripts for common monitoring tasks:
```
#!/bin/bash
# monitor-bot.sh
DROPLET_IP="XXX.XXX.XXX.XX"
CONTAINER="trading-bot-2h"

echo "=== Container Status ==="
ssh root@$DROPLET_IP "docker ps | grep $CONTAINER"

echo -e "\n=== Latest P&L ==="
ssh root@$DROPLET_IP "docker logs $CONTAINER 2>&1 | grep 'Cumulative P&L' | tail -1"

echo -e "\n=== Trade Count ==="
ssh root@$DROPLET_IP "docker logs $CONTAINER 2>&1 | grep 'TRADE EXECUTED' | wc -l"
```

### 8.10 Deployment Summary

The deployment of the trading system to the DigitalOcean cloud environment was successful. The use of Docker containerization ensured consistency between the development and production environments, while the `--network host` configuration simplified connectivity to the existing IB Gateway container. The deployment process is fully documented here and reproducible, enabling quick iterations and modifications to the trading system.

Key achievements:

- __Autonomous operation:__ Trading bot runs independently of local machine
- __Reliable connectivity:__ Successful connection to IB Gateway via localhost
- __Remote monitoring:__ Comprehensive SSH-based monitoring capabilities
- __Data persistence:__ Trade logs and results preserved outside containers
- __Reproducibility:__ Complete documentation of all steps and commands

The infrastructure is now ready for extended trading sessions and can be easily replicated or modified for different trading strategies or parameters.

## 9. Results from the Cloud Trading Run

### 9.1 Performance Summary

### 9.2 Lessons Learned and Future Scalability

## 10. Conclusion

### 10.1 Key Achievements

### 10.2 Strategy Performance Analysis

### 10.3 Technical Lessons Learned

### 10.4 Future Directions

### 10.5 Final Reflections